# Deep Past - Submit Now Baseline

This is a **minimal end-to-end notebook**: data -> train -> submission.

- Uses only `train.csv` and `test.csv`
- Trains a simple TF-IDF nearest-neighbor baseline
- Creates `submission.csv` ready for Kaggle upload

In [ ]:
from __future__ import annotations

import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [ ]:
def find_file(filename: str):
    roots = [Path('/kaggle/input'), Path('../input'), Path('data/raw'), Path('.')]
    matches = []
    for root in roots:
        if root.exists():
            matches.extend(root.rglob(filename))
    if not matches:
        return None
    matches = sorted(matches, key=lambda p: (len(str(p)), str(p)))
    return matches[0]

train_path = find_file('train.csv')
test_path = find_file('test.csv')
sample_path = find_file('sample_submission.csv')

print('train.csv ->', train_path)
print('test.csv  ->', test_path)
print('sample_submission.csv ->', sample_path)

if train_path is None or test_path is None:
    raise FileNotFoundError('Could not find train.csv and test.csv. Put data in Kaggle input or data/raw.')

In [ ]:
train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print('train shape:', train_df.shape)
print('test shape :', test_df.shape)
display(train_df.head(2))
display(test_df.head(2))

In [ ]:
SUBSCRIPT_MAP = str.maketrans({
    '₀': '0', '₁': '1', '₂': '2', '₃': '3', '₄': '4',
    '₅': '5', '₆': '6', '₇': '7', '₈': '8', '₉': '9',
    'ₓ': 'x',
})

def normalize_transliteration(text: str) -> str:
    if pd.isna(text):
        return ''
    value = str(text)
    value = value.replace('Ḫ', 'H').replace('ḫ', 'h')
    value = re.sub(r'\[x\]', ' __GAP__ ', value)
    value = re.sub(r'\[\s*\.\.\.\s*\]', ' __BIG_GAP__ ', value)
    value = re.sub(r'\.\.\.', ' __BIG_GAP__ ', value)
    value = value.translate(SUBSCRIPT_MAP)
    value = re.sub(r'[!?/]', ' ', value)
    value = value.replace(':', ' ').replace('.', ' ')
    value = value.replace('[', '').replace(']', '')
    value = value.replace('<', '').replace('>', '')
    value = value.replace('__GAP__', '<gap>').replace('__BIG_GAP__', '<big_gap>')
    return re.sub(r'\s+', ' ', value).strip()

def normalize_translation(text: str) -> str:
    if pd.isna(text):
        return ''
    value = str(text).replace('<', '').replace('>', '')
    return re.sub(r'\s+', ' ', value).strip()

def first_sentence(text: str) -> str:
    value = normalize_translation(text)
    if not value:
        return 'unknown'
    parts = re.split(r'(?<=[.!?])\s+', value)
    sent = parts[0].strip() if parts else value
    if sent and sent[-1] not in '.!?':
        sent = sent + '.'
    return sent

train_df['x'] = train_df['transliteration'].map(normalize_transliteration)
train_df['y'] = train_df['translation'].map(normalize_translation)
test_df['x'] = test_df['transliteration'].map(normalize_transliteration)

print(train_df[['x', 'y']].head(2))

In [ ]:
vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 5),
    min_df=1,
    lowercase=False,
)

X_train = vectorizer.fit_transform(train_df['x'].fillna(''))
X_test = vectorizer.transform(test_df['x'].fillna(''))

nn = NearestNeighbors(n_neighbors=1, metric='cosine', algorithm='brute')
nn.fit(X_train)
distances, indices = nn.kneighbors(X_test)

retrieved = train_df.iloc[indices.ravel()]['y'].fillna('').tolist()
predictions = [first_sentence(t) for t in retrieved]

print('Generated predictions:', len(predictions))
print('Example prediction:', predictions[0] if predictions else 'N/A')

In [ ]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'translation': predictions,
})

if submission['id'].duplicated().any():
    raise ValueError('Duplicate ids in submission')
if submission['translation'].isna().any():
    raise ValueError('Null translations in submission')

submission.to_csv('submission.csv', index=False)
print('Saved submission.csv')
display(submission.head())

## Next Step (Optional)
Replace nearest-neighbor retrieval with a seq2seq model, while keeping the same preprocessing and submission block.